In [2]:
import pandas as pd

# ✅ Puoi usare uno dei seguenti dataset
# ESEMPIO 1: Dataset AG News
# from datasets import load_dataset
# dataset = load_dataset("ag_news")
# df = pd.DataFrame(dataset['train'])

# ESEMPIO 2: Dataset personalizzato CSV
# df = pd.read_csv("notizie.csv")

# 👉 Per test: dataset fittizio
texts = ["The stock market crashed today.",
         "The football team won the championship.",
         "NASA launched a new satellite.",
         "A new restaurant opened downtown."]
labels = [0, 1, 2, 3]
df = pd.DataFrame({"text": texts, "label": labels})

num_classes = len(set(labels))
print(df.head())

                                      text  label
0          The stock market crashed today.      0
1  The football team won the championship.      1
2           NASA launched a new satellite.      2
3        A new restaurant opened downtown.      3


ORa vediamo rete LSTM

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

max_len = 100
vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

X = tokenizer.texts_to_sequences(df["text"])
X = pad_sequences(X, maxlen=max_len)

y = to_categorical(df["label"], num_classes=num_classes)


train_size = int(0.8 * len(df))
X_train = X[:train_size]
X_test  = X[train_size:]
y_train = y[:train_size]
y_test  = y[train_size:]


model_lstm = Sequential()
model_lstm.add(Embedding(input_dim=vocab_size, output_dim=64))
model_lstm.add(LSTM(64))
model_lstm.add(Dense(num_classes, activation='softmax'))

model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=1)

loss_lstm, acc_lstm = model_lstm.evaluate(X_test, y_test)
print(f"Accuracy LSTM: {acc_lstm:.4f}")



Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 658ms/step - accuracy: 0.5000 - loss: 1.3785 - val_accuracy: 0.0000e+00 - val_loss: 1.3986
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 1.3557 - val_accuracy: 0.0000e+00 - val_loss: 1.4159
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 1.3310 - val_accuracy: 0.0000e+00 - val_loss: 1.4373
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 1.3031 - val_accuracy: 0.0000e+00 - val_loss: 1.4650
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 1.2707 - val_accuracy: 0.0000e+00 - val_loss: 1.5020
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0000e+00 - loss: 1.5372
Accuracy LSTM: 0.0000


ORA PROVIAMO A PREDIRRE

In [8]:
idx = 0  # puoi cambiarlo a piacere

# 1. Estrai il testo originale (se l'hai salvato da df)
text_example = df.iloc[train_size + idx]["text"]
real_label = df.iloc[train_size + idx]["label"]

# 2. Prepara l'input
seq = tokenizer.texts_to_sequences([text_example])
padded = pad_sequences(seq, maxlen=max_len)

# 3. Predici con il modello
pred_probs = model_lstm.predict(padded)
pred_label = pred_probs.argmax(axis=1)[0]

# 4. Stampa risultati
print("📰 Notizia di test:")
print(text_example)
print("\n✅ Categoria reale:", real_label, )
print("🤖 Categoria predetta:", pred_label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
📰 Notizia di test:
A new restaurant opened downtown.

✅ Categoria reale: 3
🤖 Categoria predetta: 0


ORA BERT PRE TRAINED

In [11]:
from transformers import BertTokenizer, TFBertForSequenceClassification
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

bert_model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(bert_model_name)

# Encoding testi
X_enc = tokenizer(df["text"].tolist(), truncation=True, padding=True, max_length=128, return_tensors='tf')
y_enc = tf.convert_to_tensor(df["label"].tolist())

# Train/test split
train_size = int(0.8 * len(df))
X_train_enc = {k: v[:train_size] for k, v in X_enc.items()}
X_test_enc = {k: v[train_size:] for k, v in X_enc.items()}
y_train_enc = y_enc[:train_size]
y_test_enc = y_enc[train_size:]

# Modello
model_bert = TFBertForSequenceClassification.from_pretrained(bert_model_name, num_labels=num_classes)

model_bert.compile(
    optimizer=Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model_bert.fit(X_train_enc, y_train_enc, epochs=2, batch_size=8, validation_split=0.1)
loss_bert, acc_bert = model_bert.evaluate(X_test_enc, y_test_enc)
print(f"Accuracy BERT: {acc_bert:.4f}")

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.